In [1]:
!pip install -U anthropic pandas numpy tqdm python-dotenv

     |████████████████████████████████| 832 kB 3.9 MB/s            
  Attempting uninstall: anthropic
    Found existing installation: anthropic 0.102.0
    Uninstalling anthropic-0.102.0:
      Successfully uninstalled anthropic-0.102.0


In [2]:
from __future__ import annotations

import os
import re
import gc
import json
import time
import uuid
import pickle
import hashlib
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import anthropic

load_dotenv()

# assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY. Put it in your environment or .env file."
client = anthropic.Anthropic(api_key="")

PROVIDER = "anthropic"
MODEL_NAME = "claude-sonnet-4-6"
TEMPERATURE = 1.0
PLANNING_TEMPERATURE = 0.0

EXPERIMENT_ID = "anthropic_followup_simplestrat5_g2css3"

ANTHROPIC_THINKING = None
ANTHROPIC_ENABLE_PROMPT_CACHING = False
ANTHROPIC_CACHE_CONTROL = {"type": "ephemeral"}

N_FINAL_PER_TASK_METHOD_STRATEGY = 150
N_STRATA = 5
N_PER_STRATUM = N_FINAL_PER_TASK_METHOD_STRATEGY // N_STRATA
assert N_STRATA * N_PER_STRATUM == N_FINAL_PER_TASK_METHOD_STRATEGY

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

EXPERIMENT_DATA_PATH = Path("final_analysis/input_data/experiment_data.pkl")

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / EXPERIMENT_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "baseline": DATA_ROOT / "01_loaded_baseline",
    "planning_plans": DATA_ROOT / "02_simplestrat_planning" / "plans",
    "planning_batch_inputs": DATA_ROOT / "02_simplestrat_planning" / "batch_inputs",
    "planning_manifests": DATA_ROOT / "02_simplestrat_planning" / "manifests",
    "planning_raw_outputs": DATA_ROOT / "02_simplestrat_planning" / "raw_outputs",
    "planning_parsed": DATA_ROOT / "02_simplestrat_planning" / "parsed",
    "g2_processing": DATA_ROOT / "03_g2_css_processing",
    "round2_plans": DATA_ROOT / "04_round2_new_baselines" / "plans",
    "round2_batch_inputs": DATA_ROOT / "04_round2_new_baselines" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "04_round2_new_baselines" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "04_round2_new_baselines" / "raw_outputs",
    "round2_parsed": DATA_ROOT / "04_round2_new_baselines" / "parsed",
    "compiled": DATA_ROOT / "05_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c


In [3]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(obj), f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(json_safe(record), ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 24) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def normalize_embeddings(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(f"None of these candidate columns found: {candidates}\nAvailable columns:\n{df.columns.tolist()}")


def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj


def json_safe(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)):
            return None
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if (np.isnan(val) or np.isinf(val)) else val
    if isinstance(obj, (np.ndarray,)):
        return [json_safe(x) for x in obj.tolist()]
    try:
        if pd.isna(obj) and not isinstance(obj, (list, tuple, dict)):
            return None
    except Exception:
        pass
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    return str(obj)

In [4]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_smartphone",
        "task_family": "slogan",
        "task_label": "Smartphone slogan",
        "task_prompt_key": "smartphone",
    },
    {
        "task_id": "slogan_soda",
        "task_family": "slogan",
        "task_label": "Soda slogan",
        "task_prompt_key": "soda",
    },
    {
        "task_id": "slogan_blood_donation",
        "task_family": "slogan",
        "task_label": "Blood donation slogan",
        "task_prompt_key": "blood_donation",
    },
    {
        "task_id": "aut_shoe",
        "task_family": "aut",
        "task_label": "AUT: shoe",
        "task_prompt_key": "shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "task_id": "aut_button",
        "task_family": "aut",
        "task_label": "AUT: button",
        "task_prompt_key": "button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "task_id": "aut_key",
        "task_family": "aut",
        "task_label": "AUT: key",
        "task_prompt_key": "key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "task_id": "aut_wooden_pencil",
        "task_family": "aut",
        "task_label": "AUT: wooden pencil",
        "task_prompt_key": "wooden_pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "task_id": "aut_automobile_tire",
        "task_family": "aut",
        "task_label": "AUT: automobile tire",
        "task_prompt_key": "automobile_tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
    {
        "task_id": "story_jungle",
        "task_family": "story",
        "task_label": "Story: jungle",
        "task_prompt_key": "jungle",
    },
    {
        "task_id": "story_parachute",
        "task_family": "story",
        "task_label": "Story: parachute",
        "task_prompt_key": "parachute",
    },
    {
        "task_id": "story_horror",
        "task_family": "story",
        "task_label": "Story: horror",
        "task_prompt_key": "horror",
    },
    {
        "task_id": "story_life_last_seconds",
        "task_family": "story",
        "task_label": "Story: life / last seconds",
        "task_prompt_key": "life_last_seconds",
    },
]

TASK_BY_ID = {t["task_id"]: t for t in TASK_SETTINGS}
TASK_ORDER = [t["task_id"] for t in TASK_SETTINGS]
STRATEGIES = ["vanilla", "diverge"]

SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_shoe", "aut_button", "aut_key", "aut_wooden_pencil", "aut_automobile_tire"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def round2_final_line_for_context(strategy: str, context_type: str) -> str:
    if strategy == "vanilla":
        return "Now generate one new response for the same task."

    if strategy == "diverge" and context_type == "prior_responses":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    if strategy == "diverge" and context_type == "stratum":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from other responses that might be generated for this same task while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy/context_type: {strategy}, {context_type}")

In [5]:
run_config = {
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "temperature": TEMPERATURE,
    "planning_temperature": PLANNING_TEMPERATURE,
    "anthropic_thinking": ANTHROPIC_THINKING,
    "anthropic_enable_prompt_caching": ANTHROPIC_ENABLE_PROMPT_CACHING,
    "anthropic_cache_control": ANTHROPIC_CACHE_CONTROL if ANTHROPIC_ENABLE_PROMPT_CACHING else None,
    "n_final_per_task_method_strategy": N_FINAL_PER_TASK_METHOD_STRATEGY,
    "n_strata": N_STRATA,
    "n_per_stratum": N_PER_STRATUM,
    "task_order": TASK_ORDER,
    "strategies": STRATEGIES,
    "data_root": str(DATA_ROOT),
    "experiment_data_path": str(EXPERIMENT_DATA_PATH),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/00_metadata/experiment_config__20260523_134918__069dc09c.json')

In [6]:
@dataclass
class ExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None


@dataclass
class ProviderExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None
    provider: Optional[str] = None
    provider_label: Optional[str] = None
    model: Optional[str] = None


class GenericPicklePlaceholder:
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["state"] = state


class CompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__":
            if name == "ExperimentData":
                return ExperimentData
            if name == "ProviderExperimentData":
                return ProviderExperimentData
            return GenericPicklePlaceholder
        return super().find_class(module, name)


def load_experiment_data(path: Path) -> ExperimentData:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run the final analysis setup first, or update EXPERIMENT_DATA_PATH."
        )

    with open(path, "rb") as f:
        obj = CompatibleUnpickler(f).load()

    if hasattr(obj, "long_df") and hasattr(obj, "embeddings") and obj.long_df is not None and obj.embeddings is not None:
        return obj

    candidate_attrs = getattr(obj, "__dict__", {})
    provider_objects = []

    for attr_value in candidate_attrs.values():
        if isinstance(attr_value, dict):
            provider_objects.extend(
                [
                    v for v in attr_value.values()
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )
        elif isinstance(attr_value, (list, tuple)):
            provider_objects.extend(
                [
                    v for v in attr_value
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )

    if provider_objects:
        long_parts = [p.long_df for p in provider_objects]
        emb_parts = [np.asarray(p.embeddings) for p in provider_objects]
        combined = ExperimentData(
            long_df=pd.concat(long_parts, ignore_index=True, sort=False),
            embeddings=np.vstack(emb_parts),
        )
        if len(combined.long_df) != combined.embeddings.shape[0]:
            raise RuntimeError("Concatenated long_df and embeddings row counts do not match.")
        return combined

    raise RuntimeError("Loaded pickle object does not contain usable long_df and embeddings.")


experiment_data = load_experiment_data(EXPERIMENT_DATA_PATH)

long_df = experiment_data.long_df.copy()
long_df["_row_pos"] = np.arange(len(long_df))

if len(long_df) != experiment_data.embeddings.shape[0]:
    raise RuntimeError(
        f"long_df rows and embeddings rows do not match: {len(long_df)} vs {experiment_data.embeddings.shape[0]}"
    )

TEXT_COL = first_existing_col(
    long_df,
    ["text", "response_text", "output_text", "model_output", "clean_text", "idea", "response"],
)

print("Loaded experiment data:")
print("long_df:", long_df.shape)
print("embeddings:", experiment_data.embeddings.shape)
print("Using text column:", TEXT_COL)
print("Providers:", sorted(long_df["provider"].astype(str).unique()))

Loaded experiment data:
long_df: (64800, 53)
embeddings: (64800, 768)
Using text column: text
Providers: ['anthropic', 'gemini', 'openai']


In [7]:
baseline_mask = (
    long_df["provider"].astype(str).eq(PROVIDER)
    & long_df["round"].astype(int).eq(1)
    & long_df["strategy"].astype(str).eq("vanilla")
    & long_df["condition"].astype(str).eq("base")
    & long_df["task_id"].astype(str).isin(TASK_ORDER)
)

baseline_df = long_df.loc[baseline_mask].copy()

baseline_df["baseline_text"] = baseline_df[TEXT_COL].map(clean_model_text)
baseline_df = baseline_df.sort_values(["task_id", "group_id", "agent_id"]).reset_index(drop=True)

counts = (
    baseline_df
    .groupby("task_id", observed=True)
    .agg(
        n=("baseline_text", "size"),
        n_nonempty=("baseline_text", lambda x: x.notna().sum()),
        task_family=("task_family", "first"),
    )
    .reset_index()
    .sort_values("task_id")
)

display(counts)

missing_tasks = sorted(set(TASK_ORDER) - set(counts["task_id"]))
bad_counts = counts[counts["n"] != N_FINAL_PER_TASK_METHOD_STRATEGY]

if missing_tasks:
    raise RuntimeError(f"Missing tasks in baseline data: {missing_tasks}")

if not bad_counts.empty:
    raise RuntimeError(f"Expected 150 baseline rows per task. Bad counts:\n{bad_counts}")

if baseline_df["baseline_text"].isna().any() or baseline_df["baseline_text"].eq("").any():
    bad = baseline_df[baseline_df["baseline_text"].isna() | baseline_df["baseline_text"].eq("")]
    raise RuntimeError(f"Empty baseline text rows found:\n{bad.head()}")

baseline_light_cols = [
    "_row_pos", "provider", "provider_label", "model", "round", "task_id", "task_label",
    "task_family", "task_family_label", "strategy", "condition", "group_id", "agent_id",
    "agent_index", "baseline_text"
]
baseline_light_cols = [c for c in baseline_light_cols if c in baseline_df.columns]

baseline_light = baseline_df[baseline_light_cols].copy()

baseline_pkl_path = DIRS["baseline"] / "anthropic_neutral_r1_base_12tasks_150each.pkl"
baseline_csv_path = DIRS["baseline"] / "anthropic_neutral_r1_base_12tasks_150each.csv"

baseline_light.to_pickle(baseline_pkl_path)
baseline_light.to_csv(baseline_csv_path, index=False)

print("Saved:")
print(baseline_pkl_path)
print(baseline_csv_path)

,task_id,n,n_nonempty,task_family
0,slogan_smartphone,150,150,slogan
1,slogan_soda,150,150,slogan
2,slogan_blood_donation,150,150,slogan
3,aut_shoe,150,150,aut
4,aut_button,150,150,aut
5,aut_key,150,150,aut
6,aut_wooden_pencil,150,150,aut
7,aut_automobile_tire,150,150,aut
8,story_jungle,150,150,story
9,story_parachute,150,150,story


Saved:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/01_loaded_baseline/anthropic_neutral_r1_base_12tasks_150each.pkl
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/01_loaded_baseline/anthropic_neutral_r1_base_12tasks_150each.csv


In [8]:
PLANNING_SYSTEM_INSTRUCTIONS = (
    "You identify semantic diversity strata for controlled text-generation experiments. "
    "Return valid JSON only. Do not include markdown fences or commentary."
)


def build_simplestrat_planning_prompt(task: dict) -> str:
    return f"""
We will later generate 150 independent responses to the following task.

Task prompt:
{base_task_prompt(task)}

Identify exactly {N_STRATA} mutually distinct semantic strata for valid responses to this task.

Use the following procedure internally before choosing the final strata:
1. Consider questions that would separate the space of possible valid responses into broad, meaningfully different groups.
2. Prefer distinctions that would split the possible valid responses into reasonably balanced groups, rather than isolating rare edge cases.
3. Convert the best distinctions into categorical conceptual directions for generation.
4. Exclude distinctions based only on superficial wording, tone, length, punctuation, formatting, or synonyms.
5. Exclude strata that name a specific candidate answer, force a specific phrase, or make the original task harder to satisfy.

The final strata must satisfy all of these requirements:
- Each stratum must be a semantic/content direction, not a superficial style change.
- Each stratum must be broad enough to support many different valid responses.
- The strata must be mutually distinct enough that responses generated under different strata are likely to differ conceptually.
- The strata must collectively cover a wide range of plausible valid responses to the task.
- The generation_instruction must be concise and usable as an added constraint in a later generation prompt.

Return JSON only with this exact structure:
{{
  "task_id": "...",
  "strata": [
    {{
      "stratum_id": 1,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }}
  ]
}}

The JSON must contain exactly {N_STRATA} strata with stratum_id values 1 through {N_STRATA}.
Do not include any text outside the JSON.
""".strip()


def build_planning_plan() -> pd.DataFrame:
    rows = []
    for task in TASK_SETTINGS:
        request_basis = {
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "experiment_id": EXPERIMENT_ID,
            "stage": "simplestrat_planning",
            "task_id": task["task_id"],
            "task_family": task["task_family"],
            "task_label": task["task_label"],
            "n_strata_requested": N_STRATA,
        }
        request_key = "simplestrat_plan__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

        rows.append({
            **request_basis,
            "request_key": request_key,
            "system_instructions": PLANNING_SYSTEM_INSTRUCTIONS,
            "user_prompt": build_simplestrat_planning_prompt(task),
            "temperature": PLANNING_TEMPERATURE,
            "max_output_tokens": 900,
            "created_at_utc": now_iso(),
        })

    out = pd.DataFrame(rows)
    if out["request_key"].duplicated().any():
        raise RuntimeError("Duplicate planning request_key detected.")
    return out


planning_plan_df = build_planning_plan()

planning_plan_path = DIRS["planning_plans"] / f"simplestrat5_planning_plan__{RUN_ID}.csv"
planning_plan_df.to_csv(planning_plan_path, index=False)

print("Planning requests:", len(planning_plan_df))
display(planning_plan_df[["task_id", "task_family", "request_key"]])
print("Saved:", planning_plan_path)

Planning requests: 12


,task_id,task_family,request_key
0,slogan_smartphone,slogan,simplestrat_plan__3a9f03fb6248a1653cfdd46a
1,slogan_soda,slogan,simplestrat_plan__b7023869c69434160aa8c30f
2,slogan_blood_donation,slogan,simplestrat_plan__d00482ff32cddbb9c572f0db
3,aut_shoe,aut,simplestrat_plan__68994adc754162a862c3e4b7
4,aut_button,aut,simplestrat_plan__cc50d2a92114e86b2bb4e032
5,aut_key,aut,simplestrat_plan__54495b2ee449e5b24bcaf58c
6,aut_wooden_pencil,aut,simplestrat_plan__eacfd423146f07d21d7b74a3
7,aut_automobile_tire,aut,simplestrat_plan__42838bccbbdb59713b939618
8,story_jungle,story,simplestrat_plan__5f6b6a884449795d5cab9786
9,story_parachute,story,simplestrat_plan__7ccedb9aaf5e135c0c9da309


Saved: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/plans/simplestrat5_planning_plan__20260523_134918__069dc09c.csv


In [9]:
def make_anthropic_message_params(row: pd.Series) -> dict:
    params = {
        "model": MODEL_NAME,
        "max_tokens": int(row["max_output_tokens"]),
        "temperature": float(row["temperature"]),
        "system": row["system_instructions"],
        "messages": [
            {
                "role": "user",
                "content": row["user_prompt"],
            }
        ],
    }

    if ANTHROPIC_THINKING is not None:
        params["thinking"] = ANTHROPIC_THINKING

    if ANTHROPIC_ENABLE_PROMPT_CACHING:
        params["cache_control"] = ANTHROPIC_CACHE_CONTROL

    return params


def make_anthropic_batch_request_files(
    plan_df: pd.DataFrame,
    stage_name: str,
    batch_input_dir: Path,
    plan_dir: Path,
) -> tuple[list[dict], Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = plan_dir / f"{stem}__plan.csv"
    jsonl_path = batch_input_dir / f"{stem}__local_batch_requests.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if plan_path.exists() or jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite an existing plan or batch-input file.")

    plan_df.to_csv(plan_path, index=False)

    requests = []
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request = {
                "custom_id": row["request_key"],
                "params": make_anthropic_message_params(row),
            }
            requests.append(request)
            f.write(json.dumps(json_safe(request), ensure_ascii=False) + "\n")

    print(f"Wrote plan:          {plan_path}")
    print(f"Wrote local JSONL:   {jsonl_path}")
    print(f"Batch request count: {len(requests):,}")

    return requests, jsonl_path, plan_path


def submit_anthropic_batch(
    requests: list[dict],
    stage_name: str,
    plan_path: Path,
    local_jsonl_path: Path,
    manifest_dir: Path,
) -> dict:
    message_batch = client.messages.batches.create(requests=requests)
    batch_dump = to_jsonable(message_batch)

    batch_info = {
        "run_id": RUN_ID,
        "experiment_id": EXPERIMENT_ID,
        "stage": stage_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "message_batch": batch_dump,
        "batch_id": batch_dump.get("id"),
        "processing_status_at_submission": batch_dump.get("processing_status"),
        "submitted_at_utc": now_iso(),
        "local_jsonl_path": str(local_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = (
        manifest_dir
        / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{batch_info['batch_id']}.json"
    )

    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite manifest: {manifest_path}")

    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted Anthropic Message Batch:")
    print(json.dumps(json_safe(batch_info), indent=2))

    return batch_info


def check_anthropic_batch(batch_id: str) -> dict:
    message_batch = client.messages.batches.retrieve(batch_id)
    info = to_jsonable(message_batch)
    print(json.dumps(json_safe(info), indent=2))
    return info


def download_anthropic_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    stage_name: str,
) -> Path:
    message_batch = client.messages.batches.retrieve(batch_id)
    batch_dump = to_jsonable(message_batch)

    if batch_dump.get("processing_status") != "ended":
        raise RuntimeError(f"Batch is not ended yet. Current status: {batch_dump.get('processing_status')}")

    output_path = raw_output_dir / f"{EXPERIMENT_ID}__{stage_name}__{batch_id}__results.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output file: {output_path}")

    n = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for result in client.messages.batches.results(batch_id):
            result_dict = to_jsonable(result)
            f.write(json.dumps(json_safe(result_dict), ensure_ascii=False) + "\n")
            n += 1

    print(f"Downloaded/streamed {n:,} results to: {output_path}")
    return output_path


def extract_text_from_anthropic_message(message: dict) -> str:
    if not isinstance(message, dict):
        return ""

    texts = []
    for block in message.get("content", []) or []:
        if isinstance(block, dict) and block.get("type") == "text":
            texts.append(block.get("text", ""))

    return "\n".join(texts).strip()


def flatten_anthropic_usage(usage: Optional[dict]) -> dict:
    usage = usage or {}
    cache_creation = usage.get("cache_creation") or {}

    return {
        "usage_input_tokens": usage.get("input_tokens"),
        "usage_output_tokens": usage.get("output_tokens"),
        "usage_cache_creation_input_tokens": usage.get("cache_creation_input_tokens"),
        "usage_cache_read_input_tokens": usage.get("cache_read_input_tokens"),
        "usage_ephemeral_1h_input_tokens": cache_creation.get("ephemeral_1h_input_tokens"),
        "usage_ephemeral_5m_input_tokens": cache_creation.get("ephemeral_5m_input_tokens"),
    }


def parse_anthropic_batch_output_to_standard_files(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    stage_name: str,
    batch_id: str,
) -> dict:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0
    n_canceled = 0
    n_expired = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        result = rec.get("result", {})
        result_type = result.get("type")

        if result_type == "succeeded":
            message = result.get("message", {})
            text = clean_model_text(extract_text_from_anthropic_message(message))
            usage = message.get("usage") or {}

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": message.get("id"),
                "stop_reason": message.get("stop_reason"),
                "usage": usage,
                **flatten_anthropic_usage(usage),
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        else:
            if result_type == "errored":
                n_error += 1
            elif result_type == "canceled":
                n_canceled += 1
            elif result_type == "expired":
                n_expired += 1
            else:
                n_error += 1

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": result_type or "unknown_error",
                "text": None,
                "provider_response_id": None,
                "stop_reason": None,
                "usage": None,
                **flatten_anthropic_usage(None),
                "error": result,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, record)

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "experiment_id": EXPERIMENT_ID,
        "stage": stage_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "n_canceled": n_canceled,
        "n_expired": n_expired,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{EXPERIMENT_ID}__{stage_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(json_safe(summary), indent=2))
    return summary

In [10]:
planning_requests, planning_jsonl_path, planning_plan_path_saved = make_anthropic_batch_request_files(
    plan_df=planning_plan_df,
    stage_name="simplestrat5_planning",
    batch_input_dir=DIRS["planning_batch_inputs"],
    plan_dir=DIRS["planning_plans"],
)

planning_batch_info = submit_anthropic_batch(
    requests=planning_requests,
    stage_name="simplestrat5_planning",
    plan_path=planning_plan_path_saved,
    local_jsonl_path=planning_jsonl_path,
    manifest_dir=DIRS["planning_manifests"],
)

planning_batch_info

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/plans/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__anthropic__claude-sonnet-4-6__20260523_135028__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/batch_inputs/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__anthropic__claude-sonnet-4-6__20260523_135028__local_batch_requests.jsonl
Batch request count: 12
Submitted Anthropic Message Batch:
{
  "run_id": "20260523_134918__069dc09c",
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ",
    "archived_at": null,
    "cancel_initiated_at

{'run_id': '20260523_134918__069dc09c',
 'experiment_id': 'anthropic_followup_simplestrat5_g2css3',
 'stage': 'simplestrat5_planning',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-23T17:50:29.593725Z',
  'ended_at': None,
  'expires_at': '2026-05-24T17:50:29.593725Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 12,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-23T17:50:29.678680+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/batch_inputs/anthropic_followup_simplestrat5

In [13]:
planning_status = check_anthropic_batch(planning_batch_info["batch_id"])

{
  "id": "msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-23T17:50:29.593725Z",
  "ended_at": "2026-05-23T17:52:21.805231Z",
  "expires_at": "2026-05-24T17:50:29.593725Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 12
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ/results",
  "type": "message_batch"
}


In [14]:
planning_output_path = download_anthropic_batch_results(
    batch_id=planning_batch_info["batch_id"],
    raw_output_dir=DIRS["planning_raw_outputs"],
    stage_name="simplestrat5_planning",
)

planning_output_path

Downloaded/streamed 12 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/raw_outputs/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ__results.jsonl


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/raw_outputs/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ__results.jsonl')

In [16]:
!pip install -U json-repair

In [15]:
def parse_planning_json(text: str) -> dict:
    raw = str(text).strip()
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Could not find JSON object in planning text:\n{raw[:1000]}")

    return json.loads(match.group(0))


def build_strata_table_from_planning(parsed_pkl_path: Path) -> pd.DataFrame:
    planning_df = pd.read_pickle(parsed_pkl_path)

    rows = []
    required_fields = [
        "stratum_id",
        "name",
        "description",
        "generation_instruction",
        "why_broad",
        "why_distinct",
    ]

    for _, row in planning_df.iterrows():
        if row["status"] != "success":
            raise RuntimeError(f"Planning row failed: {row.to_dict()}")

        obj = parse_planning_json(row["text"])
        strata = obj.get("strata")

        if not isinstance(strata, list):
            raise RuntimeError(f"No strata list found for task={row['task_id']}:\n{obj}")

        if len(strata) != N_STRATA:
            raise RuntimeError(f"Expected {N_STRATA} strata for task={row['task_id']}, got {len(strata)}:\n{obj}")

        seen_ids = []
        for s in strata:
            for field in required_fields:
                if not str(s.get(field, "")).strip():
                    raise RuntimeError(
                        f"Missing or empty field `{field}` for task={row['task_id']}:\n{s}"
                    )

            sid = int(s.get("stratum_id"))
            seen_ids.append(sid)

            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": row["task_id"],
                "task_family": row["task_family"],
                "task_label": row["task_label"],
                "stratum_id": sid,
                "stratum_name": str(s.get("name", "")).strip(),
                "stratum_description": str(s.get("description", "")).strip(),
                "stratum_generation_instruction": str(s.get("generation_instruction", "")).strip(),
                "why_broad": str(s.get("why_broad", "")).strip(),
                "why_distinct": str(s.get("why_distinct", "")).strip(),
                "planning_request_key": row["request_key"],
                "planning_text": row["text"],
                "planning_batch_id": row["batch_id"],
                "created_at_utc": now_iso(),
            })

        if sorted(seen_ids) != list(range(1, N_STRATA + 1)):
            raise RuntimeError(f"Bad stratum ids for task={row['task_id']}: {seen_ids}")

    out = pd.DataFrame(rows).sort_values(["task_id", "stratum_id"]).reset_index(drop=True)
    return out


planning_parse_summary = parse_anthropic_batch_output_to_standard_files(
    batch_output_path=planning_output_path,
    plan_path=Path(planning_batch_info["plan_path"]),
    parsed_dir=DIRS["planning_parsed"],
    stage_name="simplestrat5_planning",
    batch_id=planning_batch_info["batch_id"],
)

strata_df = build_strata_table_from_planning(Path(planning_parse_summary["parsed_pkl_path"]))

strata_csv_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.csv"
strata_pkl_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.pkl"
strata_json_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.json"

strata_df.to_csv(strata_csv_path, index=False)
strata_df.to_pickle(strata_pkl_path)
write_json(strata_json_path, strata_df.to_dict(orient="records"))

print("Saved strata:")
print(strata_csv_path)
print(strata_pkl_path)
print(strata_json_path)

display(strata_df)

{
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning",
  "batch_id": "msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ",
  "n_records": 12,
  "n_success": 12,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__anthropic__claude-sonnet-4-6__msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__anthropic__claude-sonnet-4-6__msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/ant

JSONDecodeError: Expecting ',' delimiter: line 35 column 6 (char 3114)

In [17]:
from json_repair import repair_json

def extract_first_balanced_json_object(text: str) -> str:
    """
    Extract the first balanced {...} object, avoiding greedy regex problems.
    """
    s = str(text).strip()
    start = s.find("{")
    if start == -1:
        raise ValueError(f"No opening brace found:\n{s[:1000]}")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(s)):
        ch = s[i]

        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[start:i + 1]

    raise ValueError(f"Could not find balanced JSON object:\n{s[:1500]}")


def parse_planning_json_robust(text: str) -> tuple[dict, str]:
    """
    Returns (parsed_object, parse_method).
    Tries strict JSON first, then balanced extraction, then json-repair.
    """
    raw = str(text).strip()

    # Remove markdown fences if present.
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()

    # 1. Strict full-string parse.
    try:
        return json.loads(raw), "strict_full"
    except Exception:
        pass

    # 2. Strict parse of first balanced JSON object.
    candidate = extract_first_balanced_json_object(raw)
    try:
        return json.loads(candidate), "strict_balanced_object"
    except Exception:
        pass

    # 3. Repair first balanced object.
    try:
        repaired_obj = repair_json(candidate, return_objects=True)
        if isinstance(repaired_obj, dict):
            return repaired_obj, "json_repair_object"
        if isinstance(repaired_obj, str):
            return json.loads(repaired_obj), "json_repair_string"
    except Exception as e:
        raise ValueError(
            "Could not parse or repair planning JSON.\n\n"
            f"Repair error: {repr(e)}\n\n"
            f"Candidate text:\n{candidate[:3000]}"
        )


def get_existing_planning_parsed_pkl() -> Path:
    """
    Use planning_parse_summary if it exists; otherwise find the latest planning parsed pkl.
    """
    if "planning_parse_summary" in globals() and planning_parse_summary.get("parsed_pkl_path"):
        p = Path(planning_parse_summary["parsed_pkl_path"])
        if p.exists():
            return p

    matches = sorted(
        DIRS["planning_parsed"].glob(
            f"{EXPERIMENT_ID}__simplestrat5_planning__{PROVIDER}__{MODEL_NAME}__*_parsed.pkl"
        )
    )
    if not matches:
        raise FileNotFoundError("Could not find an existing parsed planning .pkl file.")
    return matches[-1]


def build_strata_table_from_planning_robust(parsed_pkl_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    planning_df = pd.read_pickle(parsed_pkl_path)

    rows = []
    audit_rows = []

    required_fields = [
        "stratum_id",
        "name",
        "description",
        "generation_instruction",
        "why_broad",
        "why_distinct",
    ]

    for _, row in planning_df.iterrows():
        if row["status"] != "success":
            raise RuntimeError(f"Planning row failed: {row.to_dict()}")

        task_id = row["task_id"]
        text = row["text"]

        try:
            obj, parse_method = parse_planning_json_robust(text)
            parse_error = None
        except Exception as e:
            audit_rows.append({
                "task_id": task_id,
                "request_key": row["request_key"],
                "parse_status": "failed",
                "parse_method": None,
                "parse_error": repr(e),
                "raw_text": text,
            })
            continue

        strata = obj.get("strata")

        audit_rows.append({
            "task_id": task_id,
            "request_key": row["request_key"],
            "parse_status": "parsed",
            "parse_method": parse_method,
            "parse_error": parse_error,
            "raw_text": text,
            "parsed_object_json": json.dumps(obj, ensure_ascii=False),
        })

        if not isinstance(strata, list):
            raise RuntimeError(f"No strata list found for task={task_id}:\n{obj}")

        if len(strata) != N_STRATA:
            raise RuntimeError(f"Expected {N_STRATA} strata for task={task_id}, got {len(strata)}:\n{obj}")

        seen_ids = []
        for s in strata:
            for field in required_fields:
                if not str(s.get(field, "")).strip():
                    raise RuntimeError(
                        f"Missing or empty field `{field}` for task={task_id}:\n{s}"
                    )

            sid = int(s.get("stratum_id"))
            seen_ids.append(sid)

            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": row["task_family"],
                "task_label": row["task_label"],
                "stratum_id": sid,
                "stratum_name": str(s.get("name", "")).strip(),
                "stratum_description": str(s.get("description", "")).strip(),
                "stratum_generation_instruction": str(s.get("generation_instruction", "")).strip(),
                "why_broad": str(s.get("why_broad", "")).strip(),
                "why_distinct": str(s.get("why_distinct", "")).strip(),
                "planning_request_key": row["request_key"],
                "planning_text": text,
                "planning_parse_method": parse_method,
                "planning_batch_id": row["batch_id"],
                "created_at_utc": now_iso(),
            })

        if sorted(seen_ids) != list(range(1, N_STRATA + 1)):
            raise RuntimeError(f"Bad stratum ids for task={task_id}: {seen_ids}")

    audit_df = pd.DataFrame(audit_rows)

    failed = audit_df[audit_df["parse_status"].eq("failed")]
    if len(failed) > 0:
        audit_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_failures.csv"
        failed.to_csv(audit_path, index=False)
        display(failed[["task_id", "request_key", "parse_error"]])
        raise RuntimeError(f"{len(failed)} planning rows could not be parsed. Saved failures to: {audit_path}")

    out = pd.DataFrame(rows).sort_values(["task_id", "stratum_id"]).reset_index(drop=True)
    return out, audit_df


planning_parsed_pkl_path = get_existing_planning_parsed_pkl()
print("Using planning parsed pkl:")
print(planning_parsed_pkl_path)

strata_df, planning_parse_audit_df = build_strata_table_from_planning_robust(planning_parsed_pkl_path)

strata_csv_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.csv"
strata_pkl_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.pkl"
strata_json_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.json"
planning_audit_csv_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_audit.csv"

strata_df.to_csv(strata_csv_path, index=False)
strata_df.to_pickle(strata_pkl_path)
write_json(strata_json_path, strata_df.to_dict(orient="records"))
planning_parse_audit_df.to_csv(planning_audit_csv_path, index=False)

print("Saved strata:")
print(strata_csv_path)
print(strata_pkl_path)
print(strata_json_path)
print("Saved parse audit:")
print(planning_audit_csv_path)

display(
    planning_parse_audit_df
    .groupby(["parse_method"], dropna=False)
    .size()
    .reset_index(name="n")
)

display(strata_df)

Using planning parsed pkl:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning__anthropic__claude-sonnet-4-6__msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ__parsed.pkl


,task_id,request_key,parse_error
1,aut_shoe,simplestrat_plan__68994adc754162a862c3e4b7,ValueError('Could not find balanced JSON objec...
5,story_life_last_seconds,simplestrat_plan__b8be388bbbaea2d71ec4da29,ValueError('Could not find balanced JSON objec...
8,aut_automobile_tire,simplestrat_plan__42838bccbbdb59713b939618,ValueError('Could not find balanced JSON objec...
11,story_parachute,simplestrat_plan__7ccedb9aaf5e135c0c9da309,ValueError('Could not find balanced JSON objec...


RuntimeError: 4 planning rows could not be parsed. Saved failures to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/simplestrat5_planning_parse_failures.csv

In [18]:
# Inspect raw text for failed planning rows

planning_failures_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_failures.csv"
planning_failures_df = pd.read_csv(planning_failures_path)

display(planning_failures_df[["task_id", "request_key", "parse_error"]])

for _, r in planning_failures_df.iterrows():
    print("\n" + "=" * 120)
    print(r["task_id"], r["request_key"])
    print("=" * 120)
    raw = str(r.get("raw_text", ""))
    print(raw[:4000])

,task_id,request_key,parse_error
0,aut_shoe,simplestrat_plan__68994adc754162a862c3e4b7,ValueError('Could not find balanced JSON objec...
1,story_life_last_seconds,simplestrat_plan__b8be388bbbaea2d71ec4da29,ValueError('Could not find balanced JSON objec...
2,aut_automobile_tire,simplestrat_plan__42838bccbbdb59713b939618,ValueError('Could not find balanced JSON objec...
3,story_parachute,simplestrat_plan__7ccedb9aaf5e135c0c9da309,ValueError('Could not find balanced JSON objec...



aut_shoe simplestrat_plan__68994adc754162a862c3e4b7
{
  "task_id": "shoe_alternative_uses",
  "strata": [
    {
      "stratum_id": 1,
      "name": "Container / Storage Vessel",
      "description": "The shoe or its hollow interior is repurposed as a container, holder, or organizer for objects, liquids, or materials.",
      "generation_instruction": "Your alternative use must involve the shoe functioning as a container, vessel, or storage/organizational tool for holding something.",
      "why_broad": "Many different items can be stored or held in a shoe—plants, utensils, small tools, ice, mail—yielding a wide variety of distinct responses.",
      "why_distinct": "This stratum focuses on passive containment, whereas other strata involve active mechanical use, structural application, or sensory/artistic purposes."
    },
    {
      "stratum_id": 2,
      "name": "Tool / Mechanical Instrument",
      "description": "The shoe or one of its physical components (sole, heel, laces, tong

In [19]:
STRICT_PLANNING_SYSTEM_INSTRUCTIONS = (
    "Return only a single valid JSON object. "
    "The response must begin with { and end with }. "
    "Do not include markdown, prose, explanation, or code fences."
)


def build_simplestrat_planning_repair_prompt(task: dict) -> str:
    return f"""
We need valid JSON for a controlled text-generation experiment.

Task prompt:
{base_task_prompt(task)}

Create exactly {N_STRATA} mutually distinct semantic strata for valid responses to this task.

Rules:
- Return only one valid JSON object.
- The first character must be {{.
- The last character must be }}.
- Do not include markdown fences.
- Do not include comments or explanation outside the JSON.
- All strings must use double quotes.
- Do not use unescaped internal double quotes inside string values.
- Each stratum must be semantic/content-based, broad, and distinct.
- Do not use strata based only on wording, tone, punctuation, formatting, or length.
- Do not include strata that force a single specific answer or phrase.

Required JSON schema:
{{
  "task_id": "{task["task_id"]}",
  "strata": [
    {{
      "stratum_id": 1,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }},
    {{
      "stratum_id": 2,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }},
    {{
      "stratum_id": 3,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }},
    {{
      "stratum_id": 4,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }},
    {{
      "stratum_id": 5,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }}
  ]
}}
""".strip()


def build_planning_repair_plan(failed_df: pd.DataFrame) -> pd.DataFrame:
    failed_task_ids = failed_df["task_id"].dropna().astype(str).tolist()
    failed_request_keys = failed_df["request_key"].dropna().astype(str).tolist()

    rows = []
    for task_id, original_request_key in zip(failed_task_ids, failed_request_keys):
        task = TASK_BY_ID[task_id]

        request_basis = {
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "experiment_id": EXPERIMENT_ID,
            "stage": "simplestrat_planning_repair",
            "task_id": task["task_id"],
            "task_family": task["task_family"],
            "task_label": task["task_label"],
            "n_strata_requested": N_STRATA,
            "original_planning_request_key": original_request_key,
            "repair_of_batch_id": planning_batch_info["batch_id"],
        }

        request_key = "repair_simplestrat_plan__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

        rows.append({
            **request_basis,
            "request_key": request_key,
            "system_instructions": STRICT_PLANNING_SYSTEM_INSTRUCTIONS,
            "user_prompt": build_simplestrat_planning_repair_prompt(task),
            "temperature": 0.0,
            "max_output_tokens": 1600,
            "created_at_utc": now_iso(),
        })

    out = pd.DataFrame(rows)

    if out["request_key"].duplicated().any():
        raise RuntimeError("Duplicate repair request_key detected.")

    return out


planning_repair_plan_df = build_planning_repair_plan(planning_failures_df)

planning_repair_plan_path = DIRS["planning_plans"] / f"simplestrat5_planning_repair_plan__{RUN_ID}.csv"
planning_repair_plan_df.to_csv(planning_repair_plan_path, index=False)

print("Repair planning requests:", len(planning_repair_plan_df))
display(planning_repair_plan_df[["task_id", "request_key", "original_planning_request_key"]])
print("Saved:", planning_repair_plan_path)

Repair planning requests: 4


,task_id,request_key,original_planning_request_key
0,aut_shoe,repair_simplestrat_plan__7e35cbfa5547b2092b556300,simplestrat_plan__68994adc754162a862c3e4b7
1,story_life_last_seconds,repair_simplestrat_plan__a54eb32a6e9cbd43b73abd14,simplestrat_plan__b8be388bbbaea2d71ec4da29
2,aut_automobile_tire,repair_simplestrat_plan__253497f1edf8c28d626a3f24,simplestrat_plan__42838bccbbdb59713b939618
3,story_parachute,repair_simplestrat_plan__d42d88e1342760958c6f35b0,simplestrat_plan__7ccedb9aaf5e135c0c9da309


Saved: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/plans/simplestrat5_planning_repair_plan__20260523_134918__069dc09c.csv


In [20]:
planning_repair_requests, planning_repair_jsonl_path, planning_repair_plan_path_saved = make_anthropic_batch_request_files(
    plan_df=planning_repair_plan_df,
    stage_name="simplestrat5_planning_repair",
    batch_input_dir=DIRS["planning_batch_inputs"],
    plan_dir=DIRS["planning_plans"],
)

planning_repair_batch_info = submit_anthropic_batch(
    requests=planning_repair_requests,
    stage_name="simplestrat5_planning_repair",
    plan_path=planning_repair_plan_path_saved,
    local_jsonl_path=planning_repair_jsonl_path,
    manifest_dir=DIRS["planning_manifests"],
)

planning_repair_batch_info

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/plans/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning_repair__anthropic__claude-sonnet-4-6__20260523_135610__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/batch_inputs/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning_repair__anthropic__claude-sonnet-4-6__20260523_135610__local_batch_requests.jsonl
Batch request count: 4
Submitted Anthropic Message Batch:
{
  "run_id": "20260523_134918__069dc09c",
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning_repair",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk",
    "archived_at": null,
    

{'run_id': '20260523_134918__069dc09c',
 'experiment_id': 'anthropic_followup_simplestrat5_g2css3',
 'stage': 'simplestrat5_planning_repair',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-23T17:56:11.323733Z',
  'ended_at': None,
  'expires_at': '2026-05-24T17:56:11.323733Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 4,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-23T17:56:11.388063+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/batch_inputs/anthropic_followup_simple

In [25]:
planning_repair_status = check_anthropic_batch(planning_repair_batch_info["batch_id"])


{
  "id": "msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-23T17:56:11.323733Z",
  "ended_at": "2026-05-23T17:57:50.787940Z",
  "expires_at": "2026-05-24T17:56:11.323733Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 4
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk/results",
  "type": "message_batch"
}


In [26]:
planning_repair_output_path = download_anthropic_batch_results(
    batch_id=planning_repair_batch_info["batch_id"],
    raw_output_dir=DIRS["planning_raw_outputs"],
    stage_name="simplestrat5_planning_repair",
)

planning_repair_parse_summary = parse_anthropic_batch_output_to_standard_files(
    batch_output_path=planning_repair_output_path,
    plan_path=Path(planning_repair_batch_info["plan_path"]),
    parsed_dir=DIRS["planning_parsed"],
    stage_name="simplestrat5_planning_repair",
    batch_id=planning_repair_batch_info["batch_id"],
)

planning_repair_df = pd.read_pickle(planning_repair_parse_summary["parsed_pkl_path"])

display(planning_repair_df["status"].value_counts(dropna=False))
display(planning_repair_df[["task_id", "status", "text"]])

Downloaded/streamed 4 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/raw_outputs/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning_repair__msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk__results.jsonl
{
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning_repair",
  "batch_id": "msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk",
  "n_records": 4,
  "n_success": 4,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/anthropic_followup_simplestrat5_g2css3__simplestrat5_planning_repair__anthropic__claude-sonnet-4-6__msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-

status
success    4
Name: count, dtype: int64

,task_id,status,text
0,aut_shoe,success,"{\n ""task_id"": ""aut_shoe"",\n ""strata"": [\n ..."
1,story_parachute,success,"{\n ""task_id"": ""story_parachute"",\n ""strata""..."
2,aut_automobile_tire,success,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str..."
3,story_life_last_seconds,success,"{\n ""task_id"": ""story_life_last_seconds"",\n ..."


In [27]:
def build_strata_table_from_mixed_planning(
    original_parsed_pkl_path: Path,
    repair_parsed_pkl_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    original_df = pd.read_pickle(original_parsed_pkl_path)
    repair_df = pd.read_pickle(repair_parsed_pkl_path)

    failed_original_keys = set(planning_failures_df["request_key"].astype(str))

    original_good = original_df[
        ~original_df["request_key"].astype(str).isin(failed_original_keys)
    ].copy()

    repair_good = repair_df[repair_df["status"].eq("success")].copy()

    if len(repair_good) != len(failed_original_keys):
        display(repair_df[["task_id", "request_key", "status", "text", "error"]])
        raise RuntimeError("Repair planning did not succeed for all failed original planning rows.")

    # Make repaired rows look like replacements for the original failed rows.
    repair_good["repair_request_key"] = repair_good["request_key"]
    repair_good["request_key"] = repair_good["original_planning_request_key"]
    repair_good["stage"] = "simplestrat_planning"
    repair_good["repaired_from_batch_id"] = repair_good["batch_id"]
    repair_good["repair_applied_at_utc"] = now_iso()

    for col in repair_good.columns:
        if col not in original_good.columns:
            original_good[col] = None

    for col in original_good.columns:
        if col not in repair_good.columns:
            repair_good[col] = None

    mixed_df = pd.concat(
        [original_good, repair_good[original_good.columns]],
        ignore_index=True,
        sort=False,
    )

    if len(mixed_df) != len(TASK_SETTINGS):
        raise RuntimeError(f"Expected {len(TASK_SETTINGS)} planning rows after repair, got {len(mixed_df)}.")

    if mixed_df["task_id"].nunique() != len(TASK_SETTINGS):
        display(mixed_df[["task_id", "request_key", "status"]].sort_values("task_id"))
        raise RuntimeError("Planning rows after repair do not contain exactly one row per task.")

    rows = []
    audit_rows = []

    required_fields = [
        "stratum_id",
        "name",
        "description",
        "generation_instruction",
        "why_broad",
        "why_distinct",
    ]

    for _, row in mixed_df.iterrows():
        if row["status"] != "success":
            raise RuntimeError(f"Planning row failed after repair: {row.to_dict()}")

        task_id = row["task_id"]
        text = row["text"]

        obj, parse_method = parse_planning_json_robust(text)
        strata = obj.get("strata")

        audit_rows.append({
            "task_id": task_id,
            "request_key": row["request_key"],
            "parse_status": "parsed",
            "parse_method": parse_method,
            "was_repaired": pd.notna(row.get("repair_request_key")),
            "repair_request_key": row.get("repair_request_key"),
            "raw_text": text,
            "parsed_object_json": json.dumps(obj, ensure_ascii=False),
        })

        if not isinstance(strata, list):
            raise RuntimeError(f"No strata list found for task={task_id}:\n{obj}")

        if len(strata) != N_STRATA:
            raise RuntimeError(f"Expected {N_STRATA} strata for task={task_id}, got {len(strata)}:\n{obj}")

        seen_ids = []
        for s in strata:
            for field in required_fields:
                if not str(s.get(field, "")).strip():
                    raise RuntimeError(
                        f"Missing or empty field `{field}` for task={task_id}:\n{s}"
                    )

            sid = int(s.get("stratum_id"))
            seen_ids.append(sid)

            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": row["task_family"],
                "task_label": row["task_label"],
                "stratum_id": sid,
                "stratum_name": str(s.get("name", "")).strip(),
                "stratum_description": str(s.get("description", "")).strip(),
                "stratum_generation_instruction": str(s.get("generation_instruction", "")).strip(),
                "why_broad": str(s.get("why_broad", "")).strip(),
                "why_distinct": str(s.get("why_distinct", "")).strip(),
                "planning_request_key": row["request_key"],
                "planning_text": text,
                "planning_parse_method": parse_method,
                "planning_batch_id": row["batch_id"],
                "planning_was_repaired": pd.notna(row.get("repair_request_key")),
                "planning_repair_request_key": row.get("repair_request_key"),
                "created_at_utc": now_iso(),
            })

        if sorted(seen_ids) != list(range(1, N_STRATA + 1)):
            raise RuntimeError(f"Bad stratum ids for task={task_id}: {seen_ids}")

    strata_out = pd.DataFrame(rows).sort_values(["task_id", "stratum_id"]).reset_index(drop=True)
    audit_out = pd.DataFrame(audit_rows).sort_values("task_id").reset_index(drop=True)

    return strata_out, audit_out


original_planning_parsed_pkl_path = planning_parsed_pkl_path
repair_planning_parsed_pkl_path = Path(planning_repair_parse_summary["parsed_pkl_path"])

strata_df, planning_parse_audit_df = build_strata_table_from_mixed_planning(
    original_parsed_pkl_path=original_planning_parsed_pkl_path,
    repair_parsed_pkl_path=repair_planning_parsed_pkl_path,
)

strata_csv_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.csv"
strata_pkl_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.pkl"
strata_json_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.json"
planning_audit_csv_path = DIRS["planning_parsed"] / "simplestrat5_planning_parse_audit_repaired.csv"

strata_df.to_csv(strata_csv_path, index=False)
strata_df.to_pickle(strata_pkl_path)
write_json(strata_json_path, strata_df.to_dict(orient="records"))
planning_parse_audit_df.to_csv(planning_audit_csv_path, index=False)

print("Saved repaired/validated strata:")
print(strata_csv_path)
print(strata_pkl_path)
print(strata_json_path)
print("Saved parse audit:")
print(planning_audit_csv_path)

display(
    planning_parse_audit_df
    .groupby(["was_repaired", "parse_method"], dropna=False)
    .size()
    .reset_index(name="n")
)

display(strata_df)

Saved repaired/validated strata:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/simplestrat5_validated_strata.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/simplestrat5_validated_strata.pkl
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/simplestrat5_validated_strata.json
Saved parse audit:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/02_simplestrat_planning/parsed/simplestrat5_planning_parse_audit_repaired.csv


,was_repaired,parse_method,n
0,False,strict_full,8
1,True,strict_full,4


,provider,model,experiment_id,task_id,task_family,task_label,stratum_id,stratum_name,stratum_description,stratum_generation_instruction,why_broad,why_distinct,planning_request_key,planning_text,planning_parse_method,planning_batch_id,planning_was_repaired,planning_repair_request_key,created_at_utc
0,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,Playground and Recreational Structure,The tire is repurposed as a component of outdo...,Focus your alternative use on how the tire can...,There are many distinct recreational applicati...,This stratum centers on active physical engage...,simplestrat_plan__42838bccbbdb59713b939618,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str...",strict_full,msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk,True,repair_simplestrat_plan__253497f1edf8c28d626a3f24,2026-05-23T17:58:38.104976+00:00
1,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,Garden and Agricultural Application,The tire or its rubber material is used in a h...,Focus your alternative use on how the tire can...,"Tires can serve as planters, raised bed border...",This stratum is grounded in growing and land m...,simplestrat_plan__42838bccbbdb59713b939618,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str...",strict_full,msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk,True,repair_simplestrat_plan__253497f1edf8c28d626a3f24,2026-05-23T17:58:38.104987+00:00
2,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,Construction and Structural Material,"The tire is used as a building block, insulati...",Focus your alternative use on how the tire con...,"Tires can be used in earthship walls, retainin...",This stratum emphasizes load-bearing or struct...,simplestrat_plan__42838bccbbdb59713b939618,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str...",strict_full,msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk,True,repair_simplestrat_plan__253497f1edf8c28d626a3f24,2026-05-23T17:58:38.104998+00:00
3,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,4,Artistic and Decorative Object,The tire or its rubber is transformed into a v...,Focus your alternative use on how the tire can...,Artists and designers can repurpose tires into...,This stratum prioritizes aesthetic and express...,simplestrat_plan__42838bccbbdb59713b939618,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str...",strict_full,msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk,True,repair_simplestrat_plan__253497f1edf8c28d626a3f24,2026-05-23T17:58:38.105008+00:00
4,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,5,Environmental and Safety Application,"The tire is used to protect ecosystems, manage...",Focus your alternative use on how the tire can...,"Tires can function as artificial reefs, dock b...",This stratum is defined by protective or ecolo...,simplestrat_plan__42838bccbbdb59713b939618,"{\n ""task_id"": ""aut_automobile_tire"",\n ""str...",strict_full,msgbatch_01LpLTi6udwaKTmu8VRQ3Zfk,True,repair_simplestrat_plan__253497f1edf8c28d626a3f24,2026-05-23T17:58:38.105019+00:00
5,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,Sensory & Tactile Tool,The button is used as a physical object to sti...,Focus on how the button's physical properties ...,"Buttons vary widely in texture, material, and ...",This stratum centers on sensory/physical inter...,simplestrat_plan__cc50d2a92114e86b2bb4e032,"{\n ""task_id"": ""button_alternative_uses"",\n ...",strict_full,msgbatch_01Vc7H7mirsbvCfcC1fdp8cZ,False,None,2026-05-23T17:58:38.103762+00:00
6,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,Artistic & Decorative Medium,The button is used as a creative or aesthetic ...,Focus on using the button as a material or med...,"Buttons come in countless shapes, 

In [28]:
def select_css_medoid_farthest(X: np.ndarray, k: int = 3) -> list[int]:
    if X.shape[0] < k:
        raise ValueError(f"Need at least {k} rows, got {X.shape[0]}")

    Xn = normalize_embeddings(X)
    sim = np.clip(Xn @ Xn.T, -1.0, 1.0)
    dist = 1.0 - sim

    avg_dist = dist.mean(axis=1)
    selected = [int(np.argmin(avg_dist))]

    while len(selected) < k:
        remaining = [i for i in range(X.shape[0]) if i not in selected]
        min_dist_to_selected = dist[np.ix_(remaining, selected)].min(axis=1)
        next_idx = remaining[int(np.argmax(min_dist_to_selected))]
        selected.append(int(next_idx))

    return selected


def build_g2_css_anchor_table(
    baseline_df: pd.DataFrame,
    embeddings: np.ndarray,
    k: int = 3,
) -> pd.DataFrame:
    rows = []

    for task_id in TASK_ORDER:
        task_df = (
            baseline_df[baseline_df["task_id"].astype(str).eq(task_id)]
            .copy()
            .sort_values(["group_id", "agent_id"])
            .reset_index(drop=True)
        )

        if len(task_df) != N_FINAL_PER_TASK_METHOD_STRATEGY:
            raise RuntimeError(f"Expected 150 baseline rows for {task_id}, got {len(task_df)}")

        row_pos = task_df["_row_pos"].to_numpy(dtype=int)
        X = embeddings[row_pos]
        selected_local = select_css_medoid_farthest(X, k=k)

        for rank, local_idx in enumerate(selected_local, start=1):
            r = task_df.iloc[local_idx].to_dict()
            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": r.get("task_family"),
                "task_label": r.get("task_label", TASK_BY_ID[task_id]["task_label"]),
                "anchor_rank": rank,
                "selection_rule": "medoid_start_farthest_first_css",
                "baseline_row_pos": int(r["_row_pos"]),
                "baseline_group_id": r.get("group_id"),
                "baseline_agent_id": r.get("agent_id"),
                "anchor_text": clean_model_text(r["baseline_text"]),
                "created_at_utc": now_iso(),
            })

    out = pd.DataFrame(rows).sort_values(["task_id", "anchor_rank"]).reset_index(drop=True)

    counts = out.groupby("task_id").size()
    if not (counts == k).all():
        raise RuntimeError(f"Bad anchor counts:\n{counts}")

    return out


g2_anchors_df = build_g2_css_anchor_table(
    baseline_df=baseline_light,
    embeddings=experiment_data.embeddings,
    k=3,
)

g2_anchor_csv_path = DIRS["g2_processing"] / "g2_css_static3_anchors.csv"
g2_anchor_pkl_path = DIRS["g2_processing"] / "g2_css_static3_anchors.pkl"
g2_anchor_json_path = DIRS["g2_processing"] / "g2_css_static3_anchors.json"

g2_anchors_df.to_csv(g2_anchor_csv_path, index=False)
g2_anchors_df.to_pickle(g2_anchor_pkl_path)
write_json(g2_anchor_json_path, g2_anchors_df.to_dict(orient="records"))

print("Saved G2/CSS anchors:")
print(g2_anchor_csv_path)
print(g2_anchor_pkl_path)
print(g2_anchor_json_path)

display(g2_anchors_df)

Saved G2/CSS anchors:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/03_g2_css_processing/g2_css_static3_anchors.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/03_g2_css_processing/g2_css_static3_anchors.pkl
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/03_g2_css_processing/g2_css_static3_anchors.json


,provider,model,experiment_id,task_id,task_family,task_label,anchor_rank,selection_rule,baseline_row_pos,baseline_group_id,baseline_agent_id,anchor_text,created_at_utc
0,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,medoid_start_farthest_first_css,34242,base_022,base_022__a1,Stack several tires vertically and fill them w...,2026-05-23T17:58:43.817726+00:00
1,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,medoid_start_farthest_first_css,34458,base_130,base_130__a1,Hung horizontally from a tree branch with a ro...,2026-05-23T17:58:43.817790+00:00
2,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,medoid_start_farthest_first_css,34262,base_032,base_032__a1,A sturdy garden planter filled with soil to gr...,2026-05-23T17:58:43.817849+00:00
3,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,medoid_start_farthest_first_css,28910,base_056,base_056__a1,Use a button as a tiny canvas for miniature pa...,2026-05-23T17:58:43.811372+00:00
4,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,medoid_start_farthest_first_css,28982,base_092,base_092__a1,A button can be glued to the end of a drawer p...,2026-05-23T17:58:43.811431+00:00
5,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,3,medoid_start_farthest_first_css,28952,base_077,base_077__a1,Slide a button under the leg of a wobbly table...,2026-05-23T17:58:43.811487+00:00
6,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,1,medoid_start_farthest_first_css,30888,base_145,base_145__a1,Using the ridged edge of a key as a makeshift ...,2026-05-23T17:58:43.813309+00:00
7,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,2,medoid_start_farthest_first_css,30734,base_068,base_068__a1,Use the serrated edge of a key to score and cu...,2026-05-23T17:58:43.813377+00:00
8,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,3,medoid_start_farthest_first_css,30702,base_052,base_052__a1,Use the serrated edge of a key to scrape and p...,2026-05-23T17:58:43.813431+00:00
9,anthropic,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_shoe,aut,AUT: shoe,1,medoid_start_farthest_first_css,27084,base_043,base_043__a1,Use the hollowed-out sole of a shoe as a small...,2026-05-23T17:58:43.809660+00:00


In [29]:
def build_simplestrat_r2_prompt(task: dict, strategy: str, stratum: dict) -> str:
    context = (
        "Conceptual direction assigned for this round:\n"
        f"{stratum['stratum_name']}: {stratum['stratum_description']}\n\n"
        "Additional generation constraint:\n"
        f"{stratum['stratum_generation_instruction']}\n\n"
        "Use this direction as the main conceptual path for the response. "
        "Do not mention the direction label or explain the direction.\n\n"
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="stratum")
    )


def build_g2_css_r2_prompt(task: dict, strategy: str, anchors: pd.DataFrame) -> str:
    anchors = anchors.sort_values("anchor_rank")
    if len(anchors) != 3:
        raise RuntimeError(f"Expected exactly 3 anchors for task={task['task_id']}, got {len(anchors)}")

    context = (
        "Previous responses from three other agents in the same first round:\n"
        f'1. "{anchors.iloc[0]["anchor_text"]}"\n'
        f'2. "{anchors.iloc[1]["anchor_text"]}"\n'
        f'3. "{anchors.iloc[2]["anchor_text"]}"\n\n'
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="prior_responses")
    )


def slot_to_stratum_id(slot_num: int) -> int:
    return ((slot_num - 1) % N_STRATA) + 1


def build_round2_new_baselines_plan(strata_df: pd.DataFrame, g2_anchors_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    strata_lookup = {
        (r.task_id, int(r.stratum_id)): r._asdict()
        for r in strata_df.itertuples(index=False)
    }

    for task in TASK_SETTINGS:
        task_id = task["task_id"]
        task_anchors = g2_anchors_df[g2_anchors_df["task_id"].astype(str).eq(task_id)].copy()

        for method in ["simplestrat5", "g2_css_static3"]:
            for strategy in STRATEGIES:
                for slot_num in range(1, N_FINAL_PER_TASK_METHOD_STRATEGY + 1):
                    slot_id = f"slot_{slot_num:03d}"

                    if method == "simplestrat5":
                        stratum_id = slot_to_stratum_id(slot_num)
                        stratum = strata_lookup[(task_id, stratum_id)]
                        user_prompt = build_simplestrat_r2_prompt(task, strategy, stratum)
                        anchor_count = 0
                        context_count = 1
                        method_label = "SimpleStrat-lite, 5 fixed auto-stratified semantic strata"
                    elif method == "g2_css_static3":
                        stratum_id = None
                        user_prompt = build_g2_css_r2_prompt(task, strategy, task_anchors)
                        anchor_count = 3
                        context_count = 3
                        method_label = "G2-inspired static CSS, 3 representative R1 examples"
                    else:
                        raise ValueError(method)

                    request_basis = {
                        "provider": PROVIDER,
                        "model": MODEL_NAME,
                        "experiment_id": EXPERIMENT_ID,
                        "round": 2,
                        "task_id": task_id,
                        "task_family": task["task_family"],
                        "task_label": task["task_label"],
                        "strategy": strategy,
                        "method": method,
                        "method_label": method_label,
                        "condition": method,
                        "slot_id": slot_id,
                        "slot_num": slot_num,
                        "stratum_id": stratum_id,
                        "anchor_count": anchor_count,
                        "context_count": context_count,
                    }

                    request_key = "r2new__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                    rows.append({
                        **request_basis,
                        "request_key": request_key,
                        "system_instructions": SYSTEM_INSTRUCTIONS,
                        "user_prompt": user_prompt,
                        "temperature": TEMPERATURE,
                        "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                        "created_at_utc": now_iso(),
                    })

    out = pd.DataFrame(rows)

    if out["request_key"].duplicated().any():
        dupes = out[out["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise RuntimeError(f"Duplicate request_key detected:\n{dupes.head()}")

    return out


round2_new_plan_df = build_round2_new_baselines_plan(
    strata_df=strata_df,
    g2_anchors_df=g2_anchors_df,
)

expected_n = len(TASK_SETTINGS) * 2 * len(STRATEGIES) * N_FINAL_PER_TASK_METHOD_STRATEGY
print("Expected R2 new-baseline requests:", expected_n)
print("Actual R2 new-baseline requests:  ", len(round2_new_plan_df))

display(
    round2_new_plan_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("request_key", "size"),
        n_strata=("stratum_id", lambda x: x.dropna().nunique()),
        n_slots=("slot_id", "nunique"),
    )
    .reset_index()
)

round2_plan_csv_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.csv"
round2_plan_pkl_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.pkl"

round2_new_plan_df.to_csv(round2_plan_csv_path, index=False)
round2_new_plan_df.to_pickle(round2_plan_pkl_path)

print("Saved:")
print(round2_plan_csv_path)
print(round2_plan_pkl_path)

Expected R2 new-baseline requests: 7200
Actual R2 new-baseline requests:   7200


,method,strategy,task_id,n,n_strata,n_slots
0,g2_css_static3,diverge,aut_automobile_tire,150,0,150
1,g2_css_static3,diverge,aut_button,150,0,150
2,g2_css_static3,diverge,aut_key,150,0,150
3,g2_css_static3,diverge,aut_shoe,150,0,150
4,g2_css_static3,diverge,aut_wooden_pencil,150,0,150
5,g2_css_static3,diverge,slogan_blood_donation,150,0,150
6,g2_css_static3,diverge,slogan_smartphone,150,0,150
7,g2_css_static3,diverge,slogan_soda,150,0,150
8,g2_css_static3,diverge,story_horror,150,0,150
9,g2_css_static3,diverge,story_jungle,150,0,150


Saved:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_134918__069dc09c.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_134918__069dc09c.pkl


In [30]:
for method in ["simplestrat5", "g2_css_static3"]:
    for strategy in ["vanilla", "diverge"]:
        ex = round2_new_plan_df[
            (round2_new_plan_df["method"] == method)
            & (round2_new_plan_df["strategy"] == strategy)
            & (round2_new_plan_df["task_id"] == "slogan_smartphone")
        ].iloc[0]

        print("\n" + "=" * 120)
        print(method, strategy, ex["request_key"])
        print("=" * 120)
        print(ex["user_prompt"])


simplestrat5 vanilla r2new__0161dee21d72aa398c72baf3
You are part of the marketing team at a tech company preparing to launch a new smartphone.

Generate exactly one marketing slogan for this brand-new smartphone.

Requirements:
- The slogan must not exceed 6 words.
- The slogan must be written in English.
- You may assume any detail about the smartphone.
- Do not list multiple slogans.
- Return only the slogan text.

Creativity goal:
- Make the response novel and appropriate for the task.

Conceptual direction assigned for this round:
Human Connection & Relationships: The slogan emphasizes how the smartphone brings people closer together, enables communication, or strengthens human bonds.

Additional generation constraint:
Your slogan should focus on how this smartphone connects people, fosters relationships, or bridges distances between humans.

Use this direction as the main conceptual path for the response. Do not mention the direction label or explain the direction.

Now generate

In [31]:
round2_requests, round2_jsonl_path, round2_plan_path_saved = make_anthropic_batch_request_files(
    plan_df=round2_new_plan_df,
    stage_name="round2_new_baselines",
    batch_input_dir=DIRS["round2_batch_inputs"],
    plan_dir=DIRS["round2_plans"],
)

round2_batch_info = submit_anthropic_batch(
    requests=round2_requests,
    stage_name="round2_new_baselines",
    plan_path=round2_plan_path_saved,
    local_jsonl_path=round2_jsonl_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/plans/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__anthropic__claude-sonnet-4-6__20260523_135850__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/batch_inputs/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__anthropic__claude-sonnet-4-6__20260523_135850__local_batch_requests.jsonl
Batch request count: 7,200
Submitted Anthropic Message Batch:
{
  "run_id": "20260523_134918__069dc09c",
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "round2_new_baselines",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01Wfo2HGYALRrfMLqEukHD4Y",
    "archived_at": null,
    "cancel_initiated_at

{'run_id': '20260523_134918__069dc09c',
 'experiment_id': 'anthropic_followup_simplestrat5_g2css3',
 'stage': 'round2_new_baselines',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01Wfo2HGYALRrfMLqEukHD4Y',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-23T17:58:53.517797Z',
  'ended_at': None,
  'expires_at': '2026-05-24T17:58:53.517797Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 7200,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01Wfo2HGYALRrfMLqEukHD4Y',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-23T17:58:53.842587+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/batch_inputs/anthropic_followup_simplestrat

In [40]:
round2_status = check_anthropic_batch(round2_batch_info["batch_id"])

{
  "id": "msgbatch_01Wfo2HGYALRrfMLqEukHD4Y",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-23T17:58:53.517797Z",
  "ended_at": "2026-05-23T18:09:44.273725Z",
  "expires_at": "2026-05-24T17:58:53.517797Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 7200
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01Wfo2HGYALRrfMLqEukHD4Y/results",
  "type": "message_batch"
}


In [41]:
round2_output_path = download_anthropic_batch_results(
    batch_id=round2_batch_info["batch_id"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    stage_name="round2_new_baselines",
)

round2_output_path

Downloaded/streamed 7,200 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/raw_outputs/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__msgbatch_01Wfo2HGYALRrfMLqEukHD4Y__results.jsonl


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/raw_outputs/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__msgbatch_01Wfo2HGYALRrfMLqEukHD4Y__results.jsonl')

In [42]:
round2_parse_summary = parse_anthropic_batch_output_to_standard_files(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    parsed_dir=DIRS["round2_parsed"],
    stage_name="round2_new_baselines",
    batch_id=round2_batch_info["batch_id"],
)

round2_new_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])

display(round2_new_df["status"].value_counts(dropna=False))

round2_bad_df = round2_new_df[
    ~round2_new_df["status"].eq("success")
    | round2_new_df["text"].isna()
    | round2_new_df["text"].astype(str).str.strip().eq("")
].copy()

print("Bad R2 new-baseline records:", len(round2_bad_df))

display(
    round2_bad_df[[
        "request_key",
        "task_id",
        "task_family",
        "strategy",
        "method",
        "slot_id",
        "status",
        "error",
    ]]
)

if len(round2_bad_df) > 20:
    raise RuntimeError("Unexpectedly many failed records. Inspect before repair.")

{
  "experiment_id": "anthropic_followup_simplestrat5_g2css3",
  "stage": "round2_new_baselines",
  "batch_id": "msgbatch_01Wfo2HGYALRrfMLqEukHD4Y",
  "n_records": 7200,
  "n_success": 7200,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/parsed/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__anthropic__claude-sonnet-4-6__msgbatch_01Wfo2HGYALRrfMLqEukHD4Y__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/04_round2_new_baselines/parsed/anthropic_followup_simplestrat5_g2css3__round2_new_baselines__anthropic__claude-sonnet-4-6__msgbatch_01Wfo2HGYALRrfMLqEukHD4Y__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/an

status
success    7200
Name: count, dtype: int64

Bad R2 new-baseline records: 0


,request_key,task_id,task_family,strategy,method,slot_id,status,error


In [43]:
def build_anthropic_repair_requests_from_bad_records(
    bad_df: pd.DataFrame,
    original_plan_path: Path,
    repair_of_batch_id: str,
) -> tuple[list[dict], pd.DataFrame]:
    original_plan_df = pd.read_csv(original_plan_path)

    failed_keys = bad_df["request_key"].dropna().astype(str).tolist()

    repair_plan_df = original_plan_df[
        original_plan_df["request_key"].astype(str).isin(failed_keys)
    ].copy()

    if len(repair_plan_df) != len(failed_keys):
        print("Failed keys:", failed_keys)
        print("Matched repair-plan rows:", len(repair_plan_df))
        raise RuntimeError("Could not match every failed request_key to the original plan.")

    repair_plan_df["original_request_key"] = repair_plan_df["request_key"]
    repair_plan_df["request_key"] = repair_plan_df["request_key"].map(lambda x: "repair_" + str(x))
    repair_plan_df["repair_of_batch_id"] = repair_of_batch_id
    repair_plan_df["repair_created_at_utc"] = now_iso()

    repair_requests = []
    for _, row in repair_plan_df.iterrows():
        repair_requests.append({
            "custom_id": row["request_key"],
            "params": make_anthropic_message_params(row),
        })

    return repair_requests, repair_plan_df


if len(round2_bad_df) > 0:
    round2_repair_requests, round2_repair_plan_df = build_anthropic_repair_requests_from_bad_records(
        bad_df=round2_bad_df,
        original_plan_path=Path(round2_batch_info["plan_path"]),
        repair_of_batch_id=round2_batch_info["batch_id"],
    )

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

    round2_repair_plan_path = (
        DIRS["round2_plans"]
        / f"{EXPERIMENT_ID}__round2_new_baselines_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__plan.csv"
    )

    round2_repair_jsonl_path = (
        DIRS["round2_batch_inputs"]
        / f"{EXPERIMENT_ID}__round2_new_baselines_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__local_batch_requests.jsonl"
    )

    if round2_repair_plan_path.exists() or round2_repair_jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite repair files.")

    round2_repair_plan_df.to_csv(round2_repair_plan_path, index=False)

    with open(round2_repair_jsonl_path, "w", encoding="utf-8") as f:
        for req in round2_repair_requests:
            f.write(json.dumps(json_safe(req), ensure_ascii=False) + "\n")

    round2_repair_batch_info = submit_anthropic_batch(
        requests=round2_repair_requests,
        stage_name="round2_new_baselines_repair",
        plan_path=round2_repair_plan_path,
        local_jsonl_path=round2_repair_jsonl_path,
        manifest_dir=DIRS["round2_manifests"],
    )

    display(round2_repair_plan_df)
    display(round2_repair_batch_info)

else:
    round2_repair_batch_info = None
    print("No repair batch submitted.")

No repair batch submitted.


In [44]:
if round2_repair_batch_info is not None:
    round2_repair_status = check_anthropic_batch(round2_repair_batch_info["batch_id"])

    round2_repair_output_path = download_anthropic_batch_results(
        batch_id=round2_repair_batch_info["batch_id"],
        raw_output_dir=DIRS["round2_raw_outputs"],
        stage_name="round2_new_baselines_repair",
    )

    round2_repair_parse_summary = parse_anthropic_batch_output_to_standard_files(
        batch_output_path=round2_repair_output_path,
        plan_path=Path(round2_repair_batch_info["plan_path"]),
        parsed_dir=DIRS["round2_parsed"],
        stage_name="round2_new_baselines_repair",
        batch_id=round2_repair_batch_info["batch_id"],
    )

    round2_repair_df = pd.read_pickle(round2_repair_parse_summary["parsed_pkl_path"])

    print(round2_repair_df.shape)
    display(round2_repair_df["status"].value_counts(dropna=False))

    display(round2_repair_df[[
        "request_key",
        "original_request_key",
        "task_id",
        "strategy",
        "method",
        "slot_id",
        "status",
        "text",
        "error",
    ]])
else:
    round2_repair_df = pd.DataFrame()
    print("No repair batch to parse.")

No repair batch to parse.


In [45]:
def merge_round2_with_repairs(
    original_df: pd.DataFrame,
    repair_df: pd.DataFrame,
) -> pd.DataFrame:
    if repair_df.empty:
        return original_df.copy()

    repair_success = repair_df[repair_df["status"].eq("success")].copy()

    if len(repair_success) != len(repair_df):
        display(repair_df[repair_df["status"] != "success"])
        raise RuntimeError("At least one repair request failed. Inspect before proceeding.")

    if "original_request_key" not in repair_success.columns:
        raise RuntimeError("Repair dataframe is missing original_request_key.")

    repaired_df = original_df.copy()

    for _, repaired_row in repair_success.iterrows():
        original_key = repaired_row["original_request_key"]

        mask = repaired_df["request_key"].astype(str).eq(str(original_key))

        if mask.sum() != 1:
            raise RuntimeError(f"Could not uniquely locate original failed row: {original_key}")

        replacement = repaired_row.copy()
        replacement["repair_request_key"] = repaired_row["request_key"]
        replacement["request_key"] = original_key
        replacement["repaired_from_batch_id"] = repaired_row.get("batch_id")
        replacement["repair_applied_at_utc"] = now_iso()

        for col in replacement.index:
            if col not in repaired_df.columns:
                repaired_df[col] = None

        repaired_df.loc[mask, replacement.index] = replacement.values

    return repaired_df


round2_final_df = merge_round2_with_repairs(
    original_df=round2_new_df,
    repair_df=round2_repair_df,
)

print(round2_final_df.shape)
display(round2_final_df["status"].value_counts(dropna=False))

remaining_bad = round2_final_df[
    ~round2_final_df["status"].eq("success")
    | round2_final_df["text"].isna()
    | round2_final_df["text"].astype(str).str.strip().eq("")
].copy()

print("Remaining bad records:", len(remaining_bad))
display(remaining_bad[["request_key", "task_id", "strategy", "method", "slot_id", "status", "error"]])

expected_n = len(TASK_SETTINGS) * 2 * len(STRATEGIES) * N_FINAL_PER_TASK_METHOD_STRATEGY

print("Expected rows:", expected_n)
print("Actual rows:  ", len(round2_final_df))

if len(round2_final_df) != expected_n:
    raise RuntimeError("Final row count does not match expectation.")

if len(remaining_bad) > 0:
    raise RuntimeError("There are still failed or empty records after repair.")

summary = (
    round2_final_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("text", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
        n_nonempty=("text", lambda x: x.notna().sum()),
        input_tokens=("usage_input_tokens", "sum"),
        output_tokens=("usage_output_tokens", "sum"),
        cache_creation_input_tokens=("usage_cache_creation_input_tokens", "sum"),
        cache_read_input_tokens=("usage_cache_read_input_tokens", "sum"),
        ephemeral_1h_input_tokens=("usage_ephemeral_1h_input_tokens", "sum"),
        ephemeral_5m_input_tokens=("usage_ephemeral_5m_input_tokens", "sum"),
    )
    .reset_index()
)

display(summary)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

compiled_csv_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}__{timestamp}.csv"
compiled_pkl_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}__{timestamp}.pkl"
summary_csv_path = DIRS["compiled"] / f"round2_new_baselines_summary__{RUN_ID}__{timestamp}.csv"

if compiled_csv_path.exists() or compiled_pkl_path.exists() or summary_csv_path.exists():
    raise FileExistsError("Refusing to overwrite compiled output files.")

round2_final_df.to_csv(compiled_csv_path, index=False)
round2_final_df.to_pickle(compiled_pkl_path)
summary.to_csv(summary_csv_path, index=False)

print("Saved compiled outputs:")
print(compiled_csv_path)
print(compiled_pkl_path)
print(summary_csv_path)

(7200, 39)


status
success    7200
Name: count, dtype: int64

Remaining bad records: 0


,request_key,task_id,strategy,method,slot_id,status,error


Expected rows: 7200
Actual rows:   7200


,method,strategy,task_id,n,n_success,n_nonempty,input_tokens,output_tokens,cache_creation_input_tokens,cache_read_input_tokens,ephemeral_1h_input_tokens,ephemeral_5m_input_tokens
0,g2_css_static3,diverge,aut_automobile_tire,150,150,150,47400,5777,0,0,0,0
1,g2_css_static3,diverge,aut_button,150,150,150,44700,4096,0,0,0,0
2,g2_css_static3,diverge,aut_key,150,150,150,44100,4724,0,0,0,0
3,g2_css_static3,diverge,aut_shoe,150,150,150,45450,4616,0,0,0,0
4,g2_css_static3,diverge,aut_wooden_pencil,150,150,150,46200,5279,0,0,0,0
5,g2_css_static3,diverge,slogan_blood_donation,150,150,150,37500,1672,0,0,0,0
6,g2_css_static3,diverge,slogan_smartphone,150,150,150,37950,1712,0,0,0,0
7,g2_css_static3,diverge,slogan_soda,150,150,150,39000,1826,0,0,0,0
8,g2_css_static3,diverge,story_horror,150,150,150,149700,41979,0,0,0,0
9,g2_css_static3,diverge,story_jungle,150,150,150,171000,50863,0,0,0,0


Saved compiled outputs:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/05_compiled/round2_new_baselines_outputs__20260523_134918__069dc09c__20260523_141003.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/05_compiled/round2_new_baselines_outputs__20260523_134918__069dc09c__20260523_141003.pkl
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/05_compiled/round2_new_baselines_summary__20260523_134918__069dc09c__20260523_141003.csv


In [46]:
final_manifest = {
    "run_id": RUN_ID,
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "data_root": str(DATA_ROOT),
    "baseline_pkl_path": str(baseline_pkl_path),
    "baseline_csv_path": str(baseline_csv_path),
    "planning_plan_path": str(planning_plan_path),
    "planning_batch_info": planning_batch_info,
    "planning_parse_summary": planning_parse_summary,
    "strata_csv_path": str(strata_csv_path),
    "strata_pkl_path": str(strata_pkl_path),
    "strata_json_path": str(strata_json_path),
    "g2_anchor_csv_path": str(g2_anchor_csv_path),
    "g2_anchor_pkl_path": str(g2_anchor_pkl_path),
    "g2_anchor_json_path": str(g2_anchor_json_path),
    "round2_plan_csv_path": str(round2_plan_csv_path),
    "round2_plan_pkl_path": str(round2_plan_pkl_path),
    "round2_batch_info": round2_batch_info,
    "round2_repair_batch_info": round2_repair_batch_info,
    "compiled_csv_path": str(compiled_csv_path),
    "compiled_pkl_path": str(compiled_pkl_path),
    "summary_csv_path": str(summary_csv_path),
    "created_at_utc": now_iso(),
}

final_manifest_path = DIRS["metadata"] / f"final_manifest__{RUN_ID}.json"
write_json(final_manifest_path, final_manifest)

print("Saved final manifest:")
print(final_manifest_path)

Saved final manifest:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/anthropic_followup_simplestrat5_g2css3/run_20260523_134918__069dc09c/00_metadata/final_manifest__20260523_134918__069dc09c.json
